In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('/home/iof_314707035/Jasper/Reimage_price_trends/models/sweeps/US_1993_2020/I20R5/sweep_summary.csv')
df.sort_values('test_mse', ascending=False).head()

,run_name,Market,I,R,epochs,batch_size,lr,weight_decay,val_ratio,early_stop_patience,...,test_n,mean_best_val_loss,mean_best_epoch,n_stopped_early,test_mse,test_acc,test_logloss,test_positive_rate,test_prob_mean,test_prob_std
15,I20_R5_bs256_ep50_lr0p001000_wd0p000000_vr0p3_...,US_1993_2020,20,5,50,256,0.0010,0.0000,0.3,5,...,857099,0.680079,4.0,1,0.247828,0.537989,0.688835,0.517777,0.499388,0.077039
17,I20_R5_bs64_ep50_lr0p001000_wd1e-04_vr0p3_es5_...,US_1993_2020,20,5,50,64,0.0010,0.0001,0.3,5,...,857099,0.681079,4.0,1,0.247788,0.535529,0.688693,0.353155,0.477012,0.069361
14,I20_R5_bs256_ep50_lr1e-04_wd0p000000_vr0p3_es5...,US_1993_2020,20,5,50,256,0.0001,0.0000,0.3,5,...,857099,0.679951,3.0,1,0.247786,0.534711,0.688672,0.517401,0.501636,0.063462
13,I20_R5_bs128_ep50_lr1e-04_wd0p000000_vr0p3_es5...,US_1993_2020,20,5,50,128,0.0001,0.0000,0.3,5,...,857099,0.679906,4.0,1,0.247636,0.541505,0.688537,0.408885,0.485103,0.083955
16,I20_R5_bs128_ep50_lr0p001000_wd1e-04_vr0p3_es5...,US_1993_2020,20,5,50,128,0.0010,0.0001,0.3,5,...,857099,0.680493,4.0,1,0.247557,0.536916,0.688226,0.436267,0.488846,0.066408


In [19]:
from pathlib import Path
import numpy as np
import pandas as pd

import model
import portfolio_backtest as pb

root = Path("/home/iof_314707035/Jasper/Reimage_price_trends")


In [20]:
sweep_root = root / "models" / "sweeps" / "US_1993_2020" / "I5R5"

run_dirs = sorted([p for p in sweep_root.iterdir() if p.is_dir()])
for p in run_dirs:
    print(p.name)


I5_R5_bs128_ep50_lr0p001000_wd0p000000_vr0p3_es5_ens1_i5r5
I5_R5_bs128_ep50_lr0p001000_wd1e-04_vr0p3_es5_ens1_i5r5
I5_R5_bs128_ep50_lr1e-04_wd0p000000_vr0p3_es5_ens1_i5r5
I5_R5_bs128_ep50_lr1e-04_wd1e-04_vr0p3_es5_ens1_i5r5
I5_R5_bs128_ep50_lr5e-04_wd0p000000_vr0p3_es5_ens1_i5r5
I5_R5_bs128_ep50_lr5e-04_wd1e-04_vr0p3_es5_ens1_i5r5
I5_R5_bs256_ep50_lr0p001000_wd0p000000_vr0p3_es5_ens1_i5r5
I5_R5_bs256_ep50_lr0p001000_wd1e-04_vr0p3_es5_ens1_i5r5
I5_R5_bs256_ep50_lr1e-04_wd0p000000_vr0p3_es5_ens1_i5r5
I5_R5_bs256_ep50_lr1e-04_wd1e-04_vr0p3_es5_ens1_i5r5
I5_R5_bs256_ep50_lr5e-04_wd0p000000_vr0p3_es5_ens1_i5r5
I5_R5_bs256_ep50_lr5e-04_wd1e-04_vr0p3_es5_ens1_i5r5
I5_R5_bs64_ep50_lr0p001000_wd0p000000_vr0p3_es5_ens1_i5r5
I5_R5_bs64_ep50_lr0p001000_wd1e-04_vr0p3_es5_ens1_i5r5
I5_R5_bs64_ep50_lr1e-04_wd0p000000_vr0p3_es5_ens1_i5r5
I5_R5_bs64_ep50_lr1e-04_wd1e-04_vr0p3_es5_ens1_i5r5
I5_R5_bs64_ep50_lr5e-04_wd0p000000_vr0p3_es5_ens1_i5r5
I5_R5_bs64_ep50_lr5e-04_wd1e-04_vr0p3_es5_ens1_i5r5
I5_R5_b

In [24]:
target_runs = [
    p for p in run_dirs
    if "bs256" in p.name and "lr1p000000e-04".replace(".", "p") not in p.name
]
candidates = []
for p in run_dirs:
    summary_path = p / "run_summary.csv"
    if summary_path.exists():
        df = pd.read_csv(summary_path)
        row = df.iloc[0]
        if (
            int(row["I"]) == 5
            and int(row["R"]) == 5
            and int(row["batch_size"]) == 256
            and abs(float(row["lr"]) - 1e-4) < 1e-12
            and abs(float(row["weight_decay"]) ) < 1e-12
        ):
            candidates.append((p, row))

for p, row in candidates:
    print(p)
    print(row[["test_mse", "test_acc", "mean_best_val_loss"]])


/home/iof_314707035/Jasper/Reimage_price_trends/models/sweeps/US_1993_2020/I5R5/I5_R5_bs256_ep50_lr1e-04_wd0p000000_vr0p3_es5_ens1_i5r5
test_mse              0.247669
test_acc              0.537317
mean_best_val_loss     0.68031
Name: 0, dtype: object


In [25]:
run_dir = candidates[0][0]

per_seed = pd.read_csv(run_dir / "per_seed_summary.csv")
per_seed = per_seed.sort_values("best_val_loss").reset_index(drop=True)
best_ckpt = per_seed.loc[0, "best_path"]

print(best_ckpt)
per_seed


models/sweeps/US_1993_2020/I5R5/I5_R5_bs256_ep50_lr1e-04_wd0p000000_vr0p3_es5_ens1_i5r5/US_1993_2020_best_model_I5R5_seed0.pth


,seed,best_path,history_path,best_score,best_val_loss,best_epoch,stopped_early,epochs_ran
0,0,models/sweeps/US_1993_2020/I5R5/I5_R5_bs256_ep...,models/sweeps/US_1993_2020/I5R5/I5_R5_bs256_ep...,0.562638,0.68031,24,True,29


In [28]:
dataset_dir = root / "data" / "training_data" / "US_1993_2020" / "I5R5S5_week"

X_test = np.load(dataset_dir / "X_test.npy")
y_test = np.load(dataset_dir / "y_test.npy")
meta_test = pd.read_csv(dataset_dir / "meta_test.csv")

for c in ["start_date", "date", "label_end_date"]:
    if c in meta_test.columns:
        meta_test[c] = pd.to_datetime(meta_test[c], errors="coerce")


In [29]:
pred_prob = model.predict_proba_from_checkpoint(
    X_test,
    best_ckpt,
    batch_size=256,
    device="cuda",   # 沒 GPU 就改 "cpu"
)

pred_prob = np.asarray(pred_prob).reshape(-1)
pred_label = (pred_prob > 0.5).astype(int)


In [31]:
pred_df = pb.make_prediction_frame(
    meta=meta_test,
    y_true=y_test,
    pred_prob=pred_prob,
    pred_label_threshold=0.5,
)

pred_df.head()


,ticker,start_date,date,label_end_date,label,ret,I,R,sample_step,sample_freq,price_source,ending_date,y_true,pred_prob,pred_label,correct
0,10001,1996-12-27,1997-01-03,1997-01-10,1,5.727096e-07,5,5,5,week,crsp,1997-01-03,1,0.375985,0,False
1,10002,1996-12-26,1997-01-03,1997-01-30,0,-4.911340e-02,5,5,5,week,crsp,1997-01-03,0,0.299591,0,True
2,10009,1996-12-23,1997-01-03,1997-01-17,0,-1.714743e-03,5,5,5,week,crsp,1997-01-03,0,0.512212,1,False
3,10011,1996-12-27,1997-01-03,1997-01-10,0,-4.225450e-02,5,5,5,week,crsp,1997-01-03,0,0.516615,1,False
4,10012,1996-12-27,1997-01-03,1997-01-10,1,1.764721e-01,5,5,5,week,crsp,1997-01-03,1,0.470006,0,False


In [32]:
grouped_df, portfolio_returns, summary = pb.run_decile_backtest(
    pred_df,
    n_groups=10,
    R=5,
    output_dir=str(run_dir / "backtest_best_single_model"),
    prefix="i5r5_best_single_model",
)

summary


,strategy,n_periods,mean_period_return,period_volatility,annualized_return,annualized_volatility,sharpe,cumulative_return,max_drawdown,positive_period_rate
0,long_top_ret,104,0.016636,0.029479,1.296947,0.209279,4.006510,4.327837,-0.173668,0.759615
1,short_bottom_ret,104,0.019035,0.021718,1.586590,0.154184,6.222137,5.944203,-0.090733,0.884615
2,long_short_ret,104,0.035671,0.015495,4.850418,0.110003,16.343488,36.858187,0.000000,1.000000


In [33]:
portfolio_returns[["date", "long_top_ret", "short_bottom_ret", "long_short_ret"]]

,date,long_top_ret,short_bottom_ret,long_short_ret
0,1997-01-03,0.038092,-0.016331,0.021761
1,1997-01-10,0.033317,0.003547,0.036864
2,1997-01-17,0.014528,0.012686,0.027214
3,1997-01-24,0.024113,0.015010,0.039123
4,1997-01-31,0.014524,0.026052,0.040577
...,...,...,...,...
99,1998-11-27,0.005593,0.026089,0.031682
100,1998-12-04,0.005097,0.032227,0.037324
101,1998-12-11,0.014311,0.023188,0.037500
102,1998-12-18,0.025486,0.026362,0.051848
